# Version-Aware RAG: fine-tune the embedder (RAG-M6)

Runs on a **free Colab T4** in about 5-15 minutes for the pilot-sized dev split. It trains
`BAAI/bge-small-en-v1.5` with MultipleNegativesRankingLoss on the leakage-checked dev pairs and
writes `final/` with `training.json` (base model, pairs sha256, settings) to Google Drive.

**Before you start (on your PC):** after the review and `vrag benchmark pilot freeze`, run
`python -m uv run vrag index finetune-data`, then upload these three files to `MyDrive/vrag/`:

- `data/build/finetune/train_pairs.jsonl`
- `data/build/finetune/train_pairs.manifest.json`
- `scripts/colab/finetune_embedder.py`

The pairs contain only permitted MIT documentation text and the benchmark questions (no restricted
release text). Runtime → Change runtime type → **T4 GPU**, then run the cells in order.

In [ ]:
# 1. Check the GPU (should list a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install the pinned library (same sentence-transformers version as the project lock)
!pip -q install "sentence-transformers==5.7.0" "datasets>=3,<5"

In [ ]:
# 3. Mount Drive and check the three uploaded files
from google.colab import drive

drive.mount("/content/drive")
import hashlib
import json
from pathlib import Path

base = Path("/content/drive/MyDrive/vrag")
pairs, manifest, script = (
    base / "train_pairs.jsonl",
    base / "train_pairs.manifest.json",
    base / "finetune_embedder.py",
)
for f in (pairs, manifest, script):
    assert f.is_file(), f"missing {f}: upload it to MyDrive/vrag/"
m = json.loads(manifest.read_text())
assert hashlib.sha256(pairs.read_bytes()).hexdigest() == m["pairs_sha256"], (
    "pairs file does not match its manifest"
)
print(
    {
        k: m[k]
        for k in (
            "pairs",
            "pairs_without_negatives",
            "dev_items",
            "held_out_items",
            "leakage_checks",
        )
    }
)

In [ ]:
# 4. Train (resumes from the latest checkpoint on Drive if the session was interrupted)
!python {script} --pairs {pairs} --out {base / "embedder-ft"} --epochs 3 --batch-size 32

In [ ]:
# 5. Sanity-check the saved model and package it for download
import shutil

import numpy as np
from sentence_transformers import SentenceTransformer

final = base / "embedder-ft" / "final"
model = SentenceTransformer(str(final), device="cuda")
q = "Represent this sentence for searching relevant passages: date_trunc on a DATE"
e = np.asarray(model.encode([q, "date_trunc truncates a date"], normalize_embeddings=True))
print("dim", e.shape[1], "cosine", round(float(e[0] @ e[1]), 4))
print(json.loads((final / "training.json").read_text()))
shutil.make_archive(str(base / "embedder-ft-final"), "zip", final)
print("download MyDrive/vrag/embedder-ft-final.zip")

## Bring the model back (on your PC)

1. Download `MyDrive/vrag/embedder-ft-final.zip`.
2. Unzip it so that `data/build/models/embedder-ft/config.json` exists (the folder is git-ignored).
3. Run:

```
python -m uv run vrag index check-model                 # provenance + load check
python -m uv run vrag index build --methods finetuned   # embeds 31,438 chunks on CPU
python -m uv run vrag eval retrieval --methods bm25,dense,finetuned,hybrid,rerank,late
```

The evaluation compares the fine-tuned model with the base dense model and every other method on
the same frozen split, with CIs and Holm-corrected differences.